This notebook evaluates the impact of individual C-fairness shilling attacks on recommender-system fairness and performance.

It prepares baseline models and fairness-aware variants (from `opt_params.json`) and applies selected attack configurations targeting unprivileged groups, as identified in the clean evaluation (`clean/metrics_top<N>.csv`).

Each attack configuration writes its own result file:

```text
results/runs/<dataset>/<attack_id>/metrics_top<N>.csv
results/runs/<dataset>/<attack_id>/ndcg.npz
results/runs/<dataset>/<attack_id>/ndcg_index.csv
```

The notebook resumes safely: set `MODEL_IDS_TO_RUN` to a subset such as `["neumf"]` to append one model without replacing prior results.

The CSV file includes:
- Precision, Recall, and NDCG computed for all users and group-specific (e.g., `ndcg_f`, `ndcg_m`, `ndcg_y`, `ndcg_o`).
- Fairness ratios for each sensitive attribute (e.g., `ndcg_gender_ratio`, `ndcg_age_ratio`).
- Metadata about the attack (`attack_id`, budget fraction, and absolute fake-user count).
- Rows cover both standard models and fairness-aware models trained per sensitive attribute.


In [ ]:

import json
import itertools
from pathlib import Path

import pandas as pd
import numpy as np

from src.helper_functions import paths
from src.helper_functions.data_loader import *
from src.helper_functions.data_splitter import *
from src.helper_functions.train_utils import *
from src.helper_functions.metrics_accuracy import *

from src.helper_functions.attack_utils import perform_attack, resolve_num_selected
from src.helper_functions.config_loader import load_attack_configs
from src.helper_functions.result_utils import (
    add_group_ratios,
    budget_to_fake_users,
    get_group_assignments,
    load_checkpoint,
    log_progress,
    make_run_key,
    mannwhitney_group_drop_test,
    store_user_labels,
    training_heartbeat,
    write_checkpoint,
)
from src.attacks.registry import CFAIR_ATTACK_REGISTRY

In [ ]:
DATASETS = ["ml-100k", "ml-1m", "lastfm-1k", "ftky", "fnyc"]
SEED = 42
RATING_THRES = 4
LIST_SIZE = 10

METRICS = {"precision": precision_at_n, "recall": recall_at_n, "ndcg": tndcg_at_n}
ALL_GROUPS_CONFIG = {"gender": ["f", "m"], "age": ["y", "o"]}
# Edit this list to run attributes separately without changing the output schema.
ATTRIBUTES_TO_RUN = ["gender", "age"]
GROUPS_CONFIG = {attr: ALL_GROUPS_CONFIG[attr] for attr in ATTRIBUTES_TO_RUN}

NUM_T_ITEMS = [10]
MIN_RATING, MAX_RATING = 1, 5
BUDGETS = [0.1]

# Use None for all models, or a subset such as ["neumf"].
MODEL_IDS_TO_RUN = None

# Nothing is trained while this is True: the run is planned and printed instead.
# Flip it to False once the plan looks right.
DRY_RUN = True

# C-fair attacks only
ATTACK_CONFIG_PATH = paths.attack_configs()
ATTACK_CONFIGS = load_attack_configs(
    ATTACK_CONFIG_PATH, "cfair_attacks", CFAIR_ATTACK_REGISTRY
)

# Any C-fair id. Print the full list with:
#   [a["attack_id"] for a in ATTACK_CONFIGS]
ATTACK_IDS_TO_RUN = [
    "cfair_influencer_push_random",
    "cfair_influencer_push_least_favorite",
    "cfair_influencer_nuke_favorite",
    "cfair_favorite_push_random",
    "cfair_favorite_push_least_favorite",
    "cfair_reverse_favorite_nuke_favorite",
]

headers = [
    "run_key", "dataset", "model", "attack_id", "attribute", "target_items", "selected_items",
    "filler_items", "budget", "fake_users", "unprivileged_group", "privileged_group",
    *METRICS.keys(),
    *[f"{m}_{g}" for groups in ALL_GROUPS_CONFIG.values() for g in groups for m in METRICS],
    *[f"{m}_{attr}_ratio" for attr in ALL_GROUPS_CONFIG for m in METRICS],
    "mannwhitney_u", "p_value",
]

PROGRESS_LOG_PATH = paths.log_file("03_train_aug.log")
HEARTBEAT_SECONDS = 60

RESUME_KEY_COLUMNS = [
    "dataset",
    "model",
    "attack_id",
    "attribute",
    "target_items",
    "selected_items",
    "filler_items",
    "budget",
]


In [ ]:
selected_model_ids = None if MODEL_IDS_TO_RUN is None else set(MODEL_IDS_TO_RUN)
selected_attacks = [
    attack for attack in ATTACK_CONFIGS
    if attack["attack_id"] in ATTACK_IDS_TO_RUN
]

unknown_attack_ids = set(ATTACK_IDS_TO_RUN) - {attack["attack_id"] for attack in ATTACK_CONFIGS}
if unknown_attack_ids:
    VANILLA_IDS = {"power_user", "bandwagon", "reverse_bandwagon"}
    hint = ""
    if unknown_attack_ids & VANILLA_IDS:
        hint = (
            " These are vanilla (group-agnostic) attacks; run them with "
            "experiments/vanilla_shilling_baselines.py, which selects the target "
            "items they require."
        )
    raise ValueError(f"Unknown attack ids: {sorted(unknown_attack_ids)}.{hint}")


def model_is_selected(model_name, model_tag=None):
    if selected_model_ids is None:
        return True
    return model_name in selected_model_ids or model_tag in selected_model_ids


for dataset in DATASETS:
    data, groups_gender, groups_age = load_dataset_by_name(dataset)
    folder = paths.dataset_dir(dataset)

    with open(paths.opt_params(dataset), "r") as f:
        opt_params = json.load(f)

    R_train, R_val, R_test, uid_to_index, iid_to_index = chronological_split_per_user(data)
    R_train_full = R_train + R_val
    log_progress(
        f"Prepared {dataset}: "
        f"train_shape={R_train.shape}, train_nnz={R_train.nnz}, "
        f"val_nnz={R_val.nnz}, test_nnz={R_test.nnz}, "
        f"train_full_nnz={R_train_full.nnz}",
        PROGRESS_LOG_PATH,
    )

    groups_map = {
        "gender": map_user_indices(groups_gender, uid_to_index),
        "age": map_user_indices(groups_age, uid_to_index),
    }

    store_user_labels(str(folder), R_test, groups_map)
    group_assignments = get_group_assignments(str(folder), LIST_SIZE, ALL_GROUPS_CONFIG)

    clean_ndcg_index = pd.read_csv(paths.pre_attack_index(dataset))
    with np.load(paths.pre_attack_ndcg(dataset), allow_pickle=True) as clean_runs:
        clean_ndcg_by_model = {
            row["model"]: clean_runs[row["key"]]
            for _, row in clean_ndcg_index.iterrows()
            if row["key"] in clean_runs.files
        }

    base_model_names = list(initialize_base_models(opt_params, SEED).keys())
    fair_model_names = list(initialize_fair_models(opt_params, SEED).keys())
    model_specs = [("base", model_name) for model_name in base_model_names] + [
        ("fair", model_name) for model_name in fair_model_names
    ]

    for attack in selected_attacks:
        attack_id = attack["attack_id"]
        post_attack_ndcg_dir = paths.post_attack_dir(dataset, attack_id)
        post_attack_ndcg_dir.mkdir(parents=True, exist_ok=True)
        result_path = paths.attack_results(dataset, attack_id, LIST_SIZE)
        ndcg_index_path = post_attack_ndcg_dir / "ndcg_index.csv"
        ndcg_path = post_attack_ndcg_dir / "ndcg.npz"

        results, ndcg_index, ndcg_store, run_id, completed = load_checkpoint(
            result_path, ndcg_index_path, ndcg_path, RESUME_KEY_COLUMNS,
            required_result_columns=["p_value"],
        )

        for attr in GROUPS_CONFIG:
            group_indices = groups_map[attr]
            if not group_indices:
                continue

            for model_kind, model_name in model_specs:
                model_tag = model_result_name(model_kind, model_name, attr)
                if not model_is_selected(model_name, model_tag):
                    continue

                group_reference_model = (
                    model_name if model_kind == "base" else get_base_model_for(model_name)
                )
                if attr not in group_assignments.get(group_reference_model, {}):
                    # FTKY and FNYC carry no age labels, so the clean run has no
                    # split for them; skip rather than abort a multi-dataset loop.
                    log_progress(
                        f"Skipping {dataset}/{group_reference_model}/{attr}: "
                        "no clean group assignment (attribute not run for this dataset)",
                        PROGRESS_LOG_PATH,
                    )
                    continue

                unpriv, priv = group_assignments[group_reference_model][attr]
                fake_user_base = len(groups_map[attr][unpriv])

                combinations = itertools.product(
                    NUM_T_ITEMS,
                    attack["num_selected_values"],
                    attack["num_filler_values"],
                    BUDGETS,
                )

                for num_targets, num_selected, num_fillers, budget in combinations:
                    use_neumf_validation = model_kind == "base" and model_name == "neumf"
                    R_train_for_attack = R_train if use_neumf_validation else R_train_full

                    # "auto" keeps fake profiles near real profile length; C-fair
                    # attacks copy unprivileged users, so the average is over them.
                    num_selected = resolve_num_selected(
                        num_selected, R_train_for_attack, num_targets, groups_map[attr][unpriv]
                    )
                    fake_users = budget_to_fake_users(budget, fake_user_base)
                    current_key = make_run_key(
                        dataset, model_tag, attack_id, attr, num_targets, num_selected, num_fillers, budget, SEED
                    )
                    if current_key in completed:
                        log_progress(
                            f"Skipping completed {dataset}/{attack_id}/{model_tag}: "
                            f"attr={attr}, selected={num_selected}, targets={num_targets}, budget={budget}",
                            PROGRESS_LOG_PATH,
                        )
                        continue

                    if DRY_RUN:
                        print(
                            f"WOULD RUN  {dataset:<10} {attack_id:<38} {model_tag:<20} "
                            f"attr={attr:<7} budget={budget} selected={num_selected} "
                            f"targets={num_targets} fake_users={fake_users}"
                        )
                        continue

                    log_progress(
                        f"Running {dataset}/{attack_id}/{model_tag}: "
                        f"attr={attr}, selected={num_selected}, targets={num_targets}, budget={budget}",
                        PROGRESS_LOG_PATH,
                    )
                    R_aug = perform_attack(
                        attack["attack_cls"], R_train_for_attack, groups_map[attr][unpriv], fake_users,
                        num_targets, num_selected, num_fillers, MIN_RATING, MAX_RATING, SEED,
                        attack["attack_config"],
                    )
                    log_progress(
                        f"Built attack matrix {dataset}/{attack_id}/{model_tag}: "
                        f"R_aug_shape={R_aug.shape}, R_aug_nnz={R_aug.nnz}",
                        PROGRESS_LOG_PATH,
                    )

                    train_label = f"post-attack {dataset}/{attack_id}/{model_tag}/attr={attr}/budget={budget}"
                    if model_kind == "base":
                        model = initialize_base_models(opt_params, SEED)[model_name]
                        log_progress(
                            f"Training {train_label}: neumf_validation={model_name == 'neumf'}, "
                            f"train_nnz={R_aug.nnz}, val_nnz={R_val.nnz if model_name == 'neumf' else 0}, "
                            f"mask_nnz={R_train_full.nnz}",
                            PROGRESS_LOG_PATH,
                        )
                        with training_heartbeat(train_label, PROGRESS_LOG_PATH, HEARTBEAT_SECONDS):
                            if model_name == "neumf":
                                R_hat = train_model(
                                    model, R_aug, R_test,
                                    val_matrix=R_val,
                                    mask_matrix=R_train_full,
                                )
                            else:
                                R_hat = train_model(model, R_aug, R_test)
                    else:
                        model = initialize_fair_models(opt_params, SEED)[model_name]
                        unprivileged_users_aug = list(groups_map[attr][unpriv]) + list(
                            range(R_test.shape[0], R_aug.shape[0])
                        )
                        log_progress(
                            f"Training {train_label}: train_nnz={R_aug.nnz}, "
                            f"unprivileged_users={len(unprivileged_users_aug)}",
                            PROGRESS_LOG_PATH,
                        )
                        with training_heartbeat(train_label, PROGRESS_LOG_PATH, HEARTBEAT_SECONDS):
                            R_hat = train_model(
                                model, R_aug, R_test,
                                unprivileged_users=unprivileged_users_aug
                            )
                    log_progress(f"Finished fit {train_label}; computing metrics", PROGRESS_LOG_PATH)

                    key = current_key
                    post_ndcg = tndcg_at_n(R_hat, R_test, RATING_THRES, LIST_SIZE)
                    ndcg_store[key] = post_ndcg
                    u_stat, p_value, _ = mannwhitney_group_drop_test(
                        clean_ndcg_by_model.get(model_tag), post_ndcg, group_indices,
                        unpriv, priv,
                    )
                    ndcg_index.append({
                        "key": key,
                        "dataset": dataset,
                        "model": model_tag,
                        "attack_id": attack_id,
                        "attribute": attr,
                        "unprivileged_group": unpriv,
                        "privileged_group": priv,
                        "target_items": num_targets,
                        "selected_items": num_selected,
                        "filler_items": num_fillers,
                        "budget": budget,
                        "fake_users": fake_users,
                    })

                    all_metrics = compute_metrics(
                        R_hat, R_test, RATING_THRES, LIST_SIZE, METRICS, groups=group_indices
                    )
                    all_metrics = add_group_ratios(all_metrics, attr, unpriv, priv, METRICS, ALL_GROUPS_CONFIG)

                    row = {
                        "run_key": current_key,
                        "dataset": dataset,
                        "model": model_tag,
                        "attack_id": attack_id,
                        "attribute": attr,
                        "unprivileged_group": unpriv,
                        "privileged_group": priv,
                        "target_items": num_targets,
                        "selected_items": num_selected,
                        "filler_items": num_fillers,
                        "budget": budget,
                        "fake_users": fake_users,
                        "mannwhitney_u": u_stat,
                        "p_value": p_value,
                    }
                    row.update(all_metrics)
                    results.append(row)
                    completed.add(current_key)

                    write_checkpoint(
                        result_path, ndcg_index_path, ndcg_path, results,
                        ndcg_index, ndcg_store, headers
                    )
                    log_progress(
                        f"Checkpointed {dataset}/{attack_id}/{model_tag}: "
                        f"attr={attr}, selected={num_selected}, targets={num_targets}, budget={budget}",
                        PROGRESS_LOG_PATH,
                    )


In [ ]:
# # Example of loading saved per-user post-attack NDCG scores for a specific run
# dataset = DATASETS[0]
# attack_id = "cfair_influencer_push_random"
# folder = f"results/{dataset}"
# post_attack_ndcg_dir = f"{folder}/{attack_id}"

# # 1) Load user labels (dataset-level)
# labels = np.load(f"{folder}/user_labels.npz", allow_pickle=True)

# # 2) Load index and select the run you want
# idx = pd.read_csv(f"{post_attack_ndcg_dir}/ndcg_index.csv")
# row = idx.query(
#     "model == 'mf-over-gender' and "
#     "attribute == 'gender' and "
#     "budget == 0.1"
# ).iloc[0]

# # 3) Load the corresponding per-user NDCG array
# runs = np.load(f"{post_attack_ndcg_dir}/ndcg.npz", allow_pickle=True)
# ndcg = runs[row["key"]]  # shape: (num_users,)

# # 4) Use the saved group metadata to slice the per-user scores
# attribute = row["attribute"]
# unprivileged_group = row["unprivileged_group"]
# privileged_group = row["privileged_group"]
# labels_for_attribute = labels[attribute]

# ndcg_unprivileged = ndcg[labels_for_attribute == unprivileged_group]
# ndcg_privileged = ndcg[labels_for_attribute == privileged_group]
